# Monte Carlo — uncertainty in plain sight

Instead of point estimates, simulate thousands of futures with random appreciation, mortgage rates, and hold periods.

Outputs:
- Distribution of true economic gains
- Probability that buying beats renting
- 5th/50th/95th percentile outcomes

This is the right way to think about a decision dominated by uncertain inputs.

In [ ]:
import sys
sys.path.insert(0, "..")

from dataclasses import replace

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from mortgage_calc import load_property, load_scenario, run_scenario

PROPERTY = "../properties/sample_lincoln_park.yaml"
SCENARIO = "../scenarios/base.yaml"
N_SIMS = 5000

prop = load_property(PROPERTY)
s = load_scenario(SCENARIO)
buy_base, rent_base, ctx_base = s["buy"], s["rent"], s["ctx"]

print(f"Property: {prop.name}")
print(f"Simulating {N_SIMS:,} futures...")

## Define your priors

These distributions encode "what you actually believe" about the future. Adjust them based on:
- Chicago condo historical data (pull it if you haven't)
- Current rate environment
- Your relocation likelihood

In [ ]:
rng = np.random.default_rng(seed=42)

# Annual appreciation: normal, centered on base, sd reflects uncertainty
appreciation_samples = rng.normal(loc=0.03, scale=0.025, size=N_SIMS)

# Mortgage rate: normal, narrow (you can lock this in)
rate_samples = rng.normal(loc=0.065, scale=0.005, size=N_SIMS)
rate_samples = np.clip(rate_samples, 0.04, 0.10)

# Hold period: weighted — your stated 5-10yr with possible 2-3yr relocation
hold_options = [2, 3, 5, 7, 10, 12]
hold_weights = [0.10, 0.15, 0.25, 0.25, 0.15, 0.10]
hold_samples = rng.choice(hold_options, size=N_SIMS, p=hold_weights)

# HOA increase: lognormal-ish, positive skew
hoa_samples = np.clip(rng.normal(loc=0.04, scale=0.02, size=N_SIMS), 0.0, 0.10)

# Show the priors
fig, axes = plt.subplots(2, 2, figsize=(11, 6))
axes[0, 0].hist(appreciation_samples * 100, bins=40, color="#534AB7")
axes[0, 0].set_title("Prior: annual appreciation (%)")
axes[0, 1].hist(rate_samples * 100, bins=40, color="#1D9E75")
axes[0, 1].set_title("Prior: mortgage rate (%)")
axes[1, 0].hist(hold_samples, bins=range(2, 14), color="#BA7517")
axes[1, 0].set_title("Prior: hold period (yrs)")
axes[1, 1].hist(hoa_samples * 100, bins=40, color="#D85A30")
axes[1, 1].set_title("Prior: annual HOA increase (%)")
plt.tight_layout()
plt.show()

## Run the simulation

In [ ]:
results = []
for i in range(N_SIMS):
    buy = replace(
        buy_base,
        annual_appreciation=appreciation_samples[i],
        mortgage_rate=rate_samples[i],
        annual_hoa_increase=hoa_samples[i],
    )
    ctx = replace(ctx_base, hold_period_years=int(hold_samples[i]))
    r = run_scenario(prop, buy, rent_base, ctx)
    results.append(r.true_economic_gain)

results = np.array(results)

prob_buy_wins = (results > 0).mean() * 100
p5, p50, p95 = np.percentile(results, [5, 50, 95])
expected = results.mean()

print(f"P(buying beats renting): {prob_buy_wins:.1f}%")
print(f"Expected gain:           ${expected:>12,.0f}")
print(f"5th percentile:          ${p5:>12,.0f}  (downside)")
print(f"50th percentile:         ${p50:>12,.0f}  (median)")
print(f"95th percentile:         ${p95:>12,.0f}  (upside)")

## Distribution of outcomes

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))
ax.hist(results / 1000, bins=60, color="#534AB7", alpha=0.85)
ax.axvline(0, color="black", linewidth=1.5, linestyle="--", label="break-even")
ax.axvline(expected / 1000, color="#D85A30", linewidth=2, label=f"expected (${expected/1000:.0f}K)")
ax.axvline(p5 / 1000, color="#888780", linewidth=1, linestyle=":", label="5th/95th pct")
ax.axvline(p95 / 1000, color="#888780", linewidth=1, linestyle=":")
ax.set_xlabel("True economic gain over hold period ($K)")
ax.set_ylabel("Count")
ax.set_title(f"Distribution of outcomes — {N_SIMS:,} simulations\nP(buy wins) = {prob_buy_wins:.0f}%")
ax.legend()
plt.tight_layout()
plt.show()

## Interpretation

- **P(buying wins) >75%:** Robust buy decision
- **P(buying wins) 50-75%:** Decent odds but real downside — consider hedges (longer commitment, larger buffer)
- **P(buying wins) <50%:** Renting is the higher-EV choice given your beliefs
- **Big tails:** Outcome is dominated by the inputs you specified — narrow the priors with research

Edit the priors above to test sensitivity. If you're 100% sure you'll stay 7+ years, that changes the picture dramatically.